# 🚀 FIXGO PRO — HUẤN LUYỆN MÔ HÌNH PHÂN LOẠI SỰ CỐ YOLOV8N-CLS (GOOGLE COLAB)

> **Đề tài**: Hệ thống dịch vụ bảo dưỡng và sửa chữa gia đình thông minh FixGo Pro  
> **Mục tiêu**: Huấn luyện mô hình Deep Learning nhận diện 5 lớp sự cố khớp với Category trong cơ sở dữ liệu (`sua-dien`, `sua-nuoc`, `dien-lanh`, `thiet-bi`, `other_unclear`).  
> **Môi trường**: Google Colab GPU (T4 / V100 / A100).

In [ ]:
# 1. Kiểm tra GPU và Cài đặt Thư viện cần thiết
!nvidia-smi
!pip install --upgrade ultralytics opencv-python matplotlib seaborn pandas scikit-learn

In [ ]:
# 2. Tải hoặc Chuẩn bị Dữ liệu Huấn luyện (Train/Val/Test: 70/20/10)
import os
import shutil
from pathlib import Path

# Clone repository hoặc upload thư mục dataset/
# !git clone https://github.com/vovanvi0906/do-an-fix-go-tho-tot.git
# DATASET_PATH = '/content/do-an-fix-go-tho-tot/ai-service/dataset'

DATASET_PATH = '/content/dataset'
print(f"Thư mục Dataset: {DATASET_PATH}")

## 🧪 THỰC NGHIỆM SO SÁNH 3 CẤU HÌNH HUẤN LUYỆN (EXPERIMENTS COMPARISON)

| Cấu hình | Kiến trúc Model | Epochs | Image Size | Batch Size | Learning Rate | Kỹ thuật Augmentation |
| :--- | :--- | :---: | :---: | :---: | :---: | :--- |
| **Config 1 (Baseline)** | `yolov8n-cls` | 30 | 224 | 16 | $0.010$ (SGD) | Mặc định |
| **Config 2 (Tuned - Khuyên dùng)** | `yolov8n-cls` | 50 | 224 | 16 | $0.005$ (Cosine LR) | Flips, Color Jitter, Mixup 0.1 |
| **Config 3 (High-Res)** | `yolov8n-cls` | 50 | 320 | 8 | $0.005$ (Cosine LR) | Flips, Random Affine, Mixup 0.1 |

In [ ]:
from ultralytics import YOLO

# 3. Huấn luyện Cấu hình 1: Baseline (30 Epochs, Image 224)
print("=== [1/3] Bắt đầu Train Config 1: Baseline ===")
model_c1 = YOLO('yolov8n-cls.pt')
results_c1 = model_c1.train(
    data=DATASET_PATH,
    epochs=30,
    imgsz=224,
    batch=16,
    lr0=0.01,
    project='runs/colab_experiments',
    name='config1_baseline',
    plots=True
)

In [ ]:
# 4. Huấn luyện Cấu hình 2: Tuned với Cosine Annealing (50 Epochs, Image 224)
print("=== [2/3] Bắt đầu Train Config 2: Tuned Cosine LR ===")
model_c2 = YOLO('yolov8n-cls.pt')
results_c2 = model_c2.train(
    data=DATASET_PATH,
    epochs=50,
    imgsz=224,
    batch=16,
    lr0=0.005,
    lrf=0.01,
    cos_lr=True,
    project='runs/colab_experiments',
    name='config2_tuned',
    plots=True
)

In [ ]:
# 5. Huấn luyện Cấu hình 3: High-Res (50 Epochs, Image 320)
print("=== [3/3] Bắt đầu Train Config 3: High-Res 320 ===")
model_c3 = YOLO('yolov8n-cls.pt')
results_c3 = model_c3.train(
    data=DATASET_PATH,
    epochs=50,
    imgsz=320,
    batch=8,
    lr0=0.005,
    cos_lr=True,
    project='runs/colab_experiments',
    name='config3_highres',
    plots=True
)

## 📊 ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP & TẠO MA TRẬN NHẦM LẪN (CONFUSION MATRIX)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import glob

# Đánh giá mô hình tốt nhất (Config 2) trên tập Test độc lập
best_model = YOLO('runs/colab_experiments/config2_tuned/weights/best.pt')
metrics = best_model.val(data=DATASET_PATH, split='test')

print(f"\n🔥 Top-1 Accuracy: {metrics.top1 * 100:.2f}%")
print(f"🔥 Top-5 Accuracy: {metrics.top5 * 100:.2f}%")

# Xuất file weights cho FixGo Backend
best_model.save('/content/yolov8n-cls-fixgo.pt')
print("✅ Đã lưu trọng số hoàn thiện tại: /content/yolov8n-cls-fixgo.pt")